In [21]:

# CELL 1 : Load cleaned data
import pandas as pd

train = pd.read_csv('train_clean.csv').dropna()
dev   = pd.read_csv('dev_clean.csv').dropna()
test  = pd.read_csv('test_clean.csv').dropna()

print('Train shape:', train.shape)
print('Columns:', train.columns.tolist())
print('\nLabel distribution:')
print(train['labels'].value_counts())
print('\ntext_clean sample:') #peek at the differences
print(train['text_clean'].iloc[0][:200])
print('\ntext_minimal sample:')
print(train['text_minimal'].iloc[0][:200]) #using 200 char only

Train shape: (6005, 4)
Columns: ['text_clean', 'text_minimal', 'labels', 'absolutist_score']

Label distribution:
labels
depressed        3751
not depressed    2254
Name: count, dtype: int64

text_clean sample:
point genuinely curious sort shit throws way actually funny i used end every previous year okay bad surely get much worse go rock bottom get fucked unimaginable ways next year point i mostly dissociat

text_minimal sample:
At this point just genuinely curious what sort of shit 2020 throws my way : It is actually funny. I used to end every previous year with "Okay, that was bad, but surely it cannot get much worse!! You 


In [22]:
# CELL 2: Helper function to evaluate any model
from sklearn.metrics import classification_report, f1_score, recall_score, precision_score

def evaluate(name, model, X_dev, y_dev):
    preds = model.predict(X_dev)
    macro_f1 = f1_score(y_dev, preds, average='macro')
    recall_dep = recall_score(y_dev, preds, pos_label='depressed')
    precision_dep = precision_score(y_dev, preds, pos_label='depressed')
    print(f'MODEL: {name}')
    print(classification_report(y_dev, preds))
    print(f'Macro F1:             {round(macro_f1, 3)}')
    print(f'Precision (depressed): {round(precision_dep, 3)}')
    print(f'Recall (depressed):    {round(recall_dep, 3)}')
    return macro_f1, precision_dep, recall_dep

In [23]:
# CELL 3 : Condition 1: Unigrams + Count Vectorizer + Logistic Regression
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

# Count Vectorizer

count_uni = CountVectorizer(
    ngram_range=(1, 1), # unigrams only
    max_features=3000,
    stop_words=None, token_pattern=r"(?u)\b\w+\b"
)

X_train_count_uni = count_uni.fit_transform(train['text_clean'])
X_dev_count_uni   = count_uni.transform(dev['text_clean'])

lr_count_uni = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_count_uni.fit(X_train_count_uni, train['labels'])

f1_1, p_1, rec_1 = evaluate(
    'Unigrams + Count Vectorizer + Logistic Regression',
    lr_count_uni, X_dev_count_uni, dev['labels']
)

MODEL: Unigrams + Count Vectorizer + Logistic Regression
               precision    recall  f1-score   support

    depressed       0.73      0.69      0.71       600
not depressed       0.57      0.62      0.59       400

     accuracy                           0.66      1000
    macro avg       0.65      0.65      0.65      1000
 weighted avg       0.67      0.66      0.66      1000

Macro F1:             0.65
Precision (depressed): 0.73
Recall (depressed):    0.685


In [24]:
# CELL 4: Condition 2: Unigrams + TF-IDF + Logistic Regression
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF Unigrams to weight words by importance
# Same vocab size as count vectorizer for fair comparison
tfidf_uni = TfidfVectorizer(
    ngram_range=(1, 1),   # unigrams only
    max_features=3000,
    stop_words=None, token_pattern=r"(?u)\b\w+\b"
)

X_train_tfidf_uni = tfidf_uni.fit_transform(train['text_clean'])
X_dev_tfidf_uni   = tfidf_uni.transform(dev['text_clean'])

lr_tfidf_uni = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_tfidf_uni.fit(X_train_tfidf_uni, train['labels'])

f1_2, p_2, rec_2 = evaluate(
    'Unigrams + TF-IDF + Logistic Regression',
    lr_tfidf_uni, X_dev_tfidf_uni, dev['labels']
)

MODEL: Unigrams + TF-IDF + Logistic Regression
               precision    recall  f1-score   support

    depressed       0.77      0.81      0.79       600
not depressed       0.70      0.65      0.67       400

     accuracy                           0.74      1000
    macro avg       0.73      0.73      0.73      1000
 weighted avg       0.74      0.74      0.74      1000

Macro F1:             0.731
Precision (depressed): 0.774
Recall (depressed):    0.812


In [25]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

count_bi = CountVectorizer(
    ngram_range=(1, 2),
    max_features=3000,
    stop_words=None, token_pattern=r"(?u)\b\w+\b"
)

X_train_count_bi = count_bi.fit_transform(train['text_clean'])
X_dev_count_bi   = count_bi.transform(dev['text_clean'])

lr_count_bi = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_count_bi.fit(X_train_count_bi, train['labels'])

f1_3, p_3, rec_3 = evaluate(
    'Bigrams + Count Vectorizer + Logistic Regression',
    lr_count_bi, X_dev_count_bi, dev['labels']
)

MODEL: Bigrams + Count Vectorizer + Logistic Regression
               precision    recall  f1-score   support

    depressed       0.74      0.69      0.71       600
not depressed       0.58      0.63      0.60       400

     accuracy                           0.67      1000
    macro avg       0.66      0.66      0.66      1000
 weighted avg       0.67      0.67      0.67      1000

Macro F1:             0.658
Precision (depressed): 0.738
Recall (depressed):    0.69


In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_bi = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=3000,
    stop_words=None, token_pattern=r"(?u)\b\w+\b"
)

X_train_tfidf_bi = tfidf_bi.fit_transform(train['text_clean'])
X_dev_tfidf_bi   = tfidf_bi.transform(dev['text_clean'])

lr_tfidf_bi = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_tfidf_bi.fit(X_train_tfidf_bi, train['labels'])

f1_4, p_4, rec_4 = evaluate(
    'Bigrams + TF-IDF + Logistic Regression',
    lr_tfidf_bi, X_dev_tfidf_bi, dev['labels']
)

MODEL: Bigrams + TF-IDF + Logistic Regression
               precision    recall  f1-score   support

    depressed       0.77      0.81      0.79       600
not depressed       0.69      0.63      0.66       400

     accuracy                           0.74      1000
    macro avg       0.73      0.72      0.72      1000
 weighted avg       0.73      0.74      0.73      1000

Macro F1:             0.722
Precision (depressed): 0.766
Recall (depressed):    0.808


In [27]:
import pandas as pd

results = pd.DataFrame({
    'Condition': [
        'Unigrams + Count Vectorizer + LR',
        'Unigrams + TF-IDF + LR',
        'Bigrams + Count Vectorizer + LR',
        'Bigrams + TF-IDF + LR'
    ],
    'Macro F1': [f1_1, f1_2, f1_3, f1_4],
    'Recall Depressed': [rec_1, rec_2, rec_3, rec_4]
})

results['Macro F1'] = results['Macro F1'].round(3)
results['Recall Depressed'] = results['Recall Depressed'].round(3)


print('VECTORIZATION COMPARISON — LOGISTIC REGRESSION ON DEV SET')

print(results.to_string(index=False))

VECTORIZATION COMPARISON — LOGISTIC REGRESSION ON DEV SET
                       Condition  Macro F1  Recall Depressed
Unigrams + Count Vectorizer + LR     0.650             0.685
          Unigrams + TF-IDF + LR     0.731             0.812
 Bigrams + Count Vectorizer + LR     0.658             0.690
           Bigrams + TF-IDF + LR     0.722             0.808


In [ ]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

svm_uni = CalibratedClassifierCV(
    LinearSVC(class_weight='balanced', max_iter=2000, random_state=42)
)
svm_uni.fit(X_train_tfidf_uni, train['labels'])

f1_svm_uni, p_svm_uni, rec_svm_uni = evaluate(
    'SVM — Unigrams + TF-IDF',
    svm_uni, X_dev_tfidf_uni, dev['labels']
)

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

# LinearSVC (faster)
# CalibratedClassifierCV so we can get confidence scores later
svm_model = CalibratedClassifierCV(
    LinearSVC(class_weight='balanced', max_iter=2000, random_state=42)
)

svm_model.fit(X_train_tfidf_bi, train['labels'])

f1_svm, p_svm, rec_svm = evaluate(
    'Bigrams + TF-IDF + SVM',
    svm_model, X_dev_tfidf_bi, dev['labels']
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_uni = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_uni.fit(X_train_tfidf_uni, train['labels'])

f1_rf_uni, p_rf_uni, rec_rf_uni = evaluate(
    'Random Forest:  Unigrams + TF-IDF',
    rf_uni, X_dev_tfidf_uni, dev['labels']
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_tfidf_bi, train['labels'])

f1_rf, p_rf, rec_rf = evaluate(
    'Bigrams + TF-IDF + Random Forest',
    rf_model, X_dev_tfidf_bi, dev['labels']
)

In [ ]:
from sklearn.naive_bayes import MultinomialNB

# Naive Bayes Condition 1: Unigrams + Count Vectorizer
nb_count_uni = MultinomialNB()
nb_count_uni.fit(X_train_count_uni, train['labels'])
f1_nb_count_uni, p_nb_count_uni, rec_nb_count_uni = evaluate(
    'Unigrams + Count Vectorizer + Naive Bayes',
    nb_count_uni, X_dev_count_uni, dev['labels']
)

# Naive Bayes Condition 2: Unigrams + TF-IDF
nb_tfidf_uni = MultinomialNB()
nb_tfidf_uni.fit(X_train_tfidf_uni, train['labels'])
f1_nb_tfidf_uni, p_nb_tfidf_uni, rec_nb_tfidf_uni = evaluate(
    'Unigrams + TF-IDF + Naive Bayes',
    nb_tfidf_uni, X_dev_tfidf_uni, dev['labels']
)

# Naive Bayes Condition 3: Bigrams + Count Vectorizer
nb_count_bi = MultinomialNB()
nb_count_bi.fit(X_train_count_bi, train['labels'])
f1_nb_count_bi, p_nb_count_bi, rec_nb_count_bi = evaluate(
    'Bigrams + Count Vectorizer + Naive Bayes',
    nb_count_bi, X_dev_count_bi, dev['labels']
)

# Naive Bayes Condition 4: Bigrams + TF-IDF
nb_tfidf_bi = MultinomialNB()
nb_tfidf_bi.fit(X_train_tfidf_bi, train['labels'])
f1_nb_tfidf_bi, p_nb_tfidf_bi, rec_nb_tfidf_bi = evaluate(
    'Bigrams + TF-IDF + Naive Bayes',
    nb_tfidf_bi, X_dev_tfidf_bi, dev['labels']
)

In [ ]:
results_full = pd.DataFrame({
    'Model': [
        'Unigrams + Count Vectorizer + LR',
        'Unigrams + TF-IDF + LR',
        'Bigrams + Count Vectorizer + LR',
        'Bigrams + TF-IDF + LR',
        'Unigrams + TF-IDF + SVM',
        'Bigrams + TF-IDF + SVM',
        'Unigrams + TF-IDF + Random Forest',
        'Bigrams + TF-IDF + Random Forest',
        'Unigrams + Count Vectorizer + Naive Bayes',
        'Unigrams + TF-IDF + Naive Bayes',
        'Bigrams + Count Vectorizer + Naive Bayes',
        'Bigrams + TF-IDF + Naive Bayes'
    ],
    'Macro F1': [
        f1_1, f1_2, f1_3, f1_4,
        f1_svm_uni, f1_svm,
        f1_rf_uni, f1_rf,
        f1_nb_count_uni, f1_nb_tfidf_uni,
        f1_nb_count_bi, f1_nb_tfidf_bi
    ],
    'Precision (Depressed)': [
        p_1, p_2, p_3, p_4,
        p_svm_uni, p_svm,
        p_rf_uni, p_rf,
        p_nb_count_uni, p_nb_tfidf_uni,
        p_nb_count_bi, p_nb_tfidf_bi
    ],
    'Recall (Depressed)': [
        rec_1, rec_2, rec_3, rec_4,
        rec_svm_uni, rec_svm,
        rec_rf_uni, rec_rf,
        rec_nb_count_uni, rec_nb_tfidf_uni,
        rec_nb_count_bi, rec_nb_tfidf_bi
    ]
})

results_full = results_full.round(3)
print('FULL MODEL COMPARISON — DEV SET')
print(results_full.to_string(index=False))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

models = {
    'Logistic Regression (Bigrams + TF-IDF)': (lr_tfidf_bi, X_dev_tfidf_bi),
    'SVM (Bigrams + TF-IDF)': (svm_model, X_dev_tfidf_bi),
    'Random Forest (Bigrams + TF-IDF)': (rf_model, X_dev_tfidf_bi),
    'Naive Bayes (Bigrams + TF-IDF)': (nb_tfidf_bi, X_dev_tfidf_bi)
}

fig, axes = plt.subplots(1, 4, figsize=(24, 5))

for ax, (name, (model, X_dev)) in zip(axes, models.items()):
    preds = model.predict(X_dev)
    cm = confusion_matrix(dev['labels'], preds,
                          labels=['depressed', 'not depressed'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                   display_labels=['depressed', 'not depressed'])
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(name, fontsize=9)

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved confusion_matrices.png')

In [ ]:
# Extract TP TN FP FN for each model
# Depressed is our positive class

def get_confusion_stats(model, X_dev, y_dev):
    preds = model.predict(X_dev)
    cm = confusion_matrix(y_dev, preds, labels=['depressed', 'not depressed'])
    TP = cm[0][0]  # predicted depressed, actually depressed
    FN = cm[0][1]  # predicted not depressed, actually depressed (WORST CASE)
    FP = cm[1][0]  # predicted depressed, actually not depressed
    TN = cm[1][1]  # predicted not depressed, actually not depressed
    return TP, FN, FP, TN


print('CONFUSION MATRIX BREAKDOWN — DEV SET')
print('\nDefinitions for this project:')
print('TP = flagged as depressed AND actually depressed (caught)')
print('FN = flagged as not depressed BUT actually depressed (MISSED)')
print('FP = flagged as depressed BUT actually not depressed (false alarm)')
print('TN = flagged as not depressed AND actually not depressed (correct clear)')

for name, (model, X_dev) in models.items():
    TP, FN, FP, TN = get_confusion_stats(model, X_dev, dev['labels'])
    print(f'\n{name}')
    print(f'  True Positives  (caught depressed):     {TP}')
    print(f'  False Negatives (MISSED depressed):     {FN}')
    print(f'  False Positives (false alarms):         {FP}')
    print(f'  True Negatives  (correctly cleared):    {TN}')

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.svm import LinearSVC
import matplotlib.pyplot as plt
import pandas as pd

y_binary = (dev['labels'] == 'depressed').astype(int)

# All twelve models: name -> (model, dev features)
all_models = {
    'LR  | Unigram Count':  (lr_count_uni,  X_dev_count_uni),
    'LR  | Unigram TF-IDF': (lr_tfidf_uni,  X_dev_tfidf_uni),
    'LR  | Bigram Count':   (lr_count_bi,   X_dev_count_bi),
    'LR  | Bigram TF-IDF':  (lr_tfidf_bi,   X_dev_tfidf_bi),
    'SVM | Unigram TF-IDF': (svm_uni,       X_dev_tfidf_uni),
    'SVM | Bigram TF-IDF':  (svm_model,     X_dev_tfidf_bi),
    'RF  | Unigram TF-IDF': (rf_uni,        X_dev_tfidf_uni),
    'RF  | Bigram TF-IDF':  (rf_model,      X_dev_tfidf_bi),
    'NB  | Unigram Count':  (nb_count_uni,  X_dev_count_uni),
    'NB  | Unigram TF-IDF': (nb_tfidf_uni,  X_dev_tfidf_uni),
    'NB  | Bigram Count':   (nb_count_bi,   X_dev_count_bi),
    'NB  | Bigram TF-IDF':  (nb_tfidf_bi,   X_dev_tfidf_bi),
}

# AUC for every model (ranking quality, independent of the cutoff)
rows = []
for name, (model, X_dev) in all_models.items():
    dep_idx = list(model.classes_).index('depressed')
    scores = model.predict_proba(X_dev)[:, dep_idx]
    rows.append((name, roc_auc_score(y_binary, scores)))

# Check: plain LinearSVC without the CalibratedClassifierCV wrapper
raw_svm = LinearSVC(class_weight='balanced', max_iter=5000, random_state=42)
raw_svm.fit(X_train_tfidf_bi, train['labels'])
raw_scores = -raw_svm.decision_function(X_dev_tfidf_bi)  # flipped so higher = depressed
rows.append(('SVM | Bigram TF-IDF (plain LinearSVC, no calibration)',
             roc_auc_score(y_binary, raw_scores)))

auc_table = pd.DataFrame(rows, columns=['Model', 'AUC']).sort_values('AUC', ascending=False)
auc_table['AUC'] = auc_table['AUC'].round(3)
print('AUC ON DEV SET')
print(auc_table.to_string(index=False))

# ROC plot: bigram TF-IDF models plus the plain SVM
plot_models = {k: v for k, v in all_models.items() if 'Bigram TF-IDF' in k}
colors = ['blue', 'red', 'green', 'orange']

fig, ax = plt.subplots(figsize=(8, 6))
for (name, (model, X_dev)), color in zip(plot_models.items(), colors):
    dep_idx = list(model.classes_).index('depressed')
    scores = model.predict_proba(X_dev)[:, dep_idx]
    fpr, tpr, _ = roc_curve(y_binary, scores)
    ax.plot(fpr, tpr, color=color,
            label=f'{name} (AUC = {roc_auc_score(y_binary, scores):.3f})')

fpr, tpr, _ = roc_curve(y_binary, raw_scores)
ax.plot(fpr, tpr, color='purple', linestyle=':',
        label=f'SVM plain (AUC = {roc_auc_score(y_binary, raw_scores):.3f})')

ax.plot([0, 1], [0, 1], 'k--', label='Random baseline (AUC = 0.5)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate (Recall)')
ax.set_title('ROC Curves, Bigram TF-IDF Models on Dev Set')
ax.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import joblib

# Bigram models
joblib.dump(lr_tfidf_bi, 'lr_model.pkl')
joblib.dump(svm_model, 'svm_model.pkl')
joblib.dump(rf_model, 'random_forest_model.pkl')
joblib.dump(nb_tfidf_bi, 'nb_model.pkl')

# Unigram models
joblib.dump(lr_tfidf_uni, 'lr_uni_model.pkl')
joblib.dump(svm_uni, 'svm_uni_model.pkl')
joblib.dump(rf_uni, 'rf_uni_model.pkl')
joblib.dump(nb_tfidf_uni, 'nb_uni_model.pkl')

# Count vectorizer models
joblib.dump(lr_count_uni, 'lr_count_uni_model.pkl')
joblib.dump(lr_count_bi, 'lr_count_bi_model.pkl')
joblib.dump(nb_count_uni, 'nb_count_uni_model.pkl')
joblib.dump(nb_count_bi, 'nb_count_bi_model.pkl')

# Vectorizers
joblib.dump(tfidf_bi, 'tfidf_bigram_vectorizer.pkl')
joblib.dump(tfidf_uni, 'tfidf_unigram_vectorizer.pkl')
joblib.dump(count_uni, 'count_uni_vectorizer.pkl')
joblib.dump(count_bi, 'count_bi_vectorizer.pkl')

print('All models and vectorizers saved.')

In [ ]:
# A peek into what my  models are getting wrong
# False Negatives = depressed posts the model missed
# False Positives = not depressed posts the model wrongly flagged

def error_analysis(model, X_dev, y_dev, df_dev, model_name, n=10):
    preds = model.predict(X_dev)

    df = df_dev.copy().reset_index(drop=True)
    df['predicted'] = preds
    df['true'] = y_dev.reset_index(drop=True)

    false_negatives = df[
        (df['true'] == 'depressed') &
        (df['predicted'] == 'not depressed')
    ]

    false_positives = df[
        (df['true'] == 'not depressed') &
        (df['predicted'] == 'depressed')
    ]


    print(f'ERROR ANALYSIS — {model_name}')
    print(f'\nTotal False Negatives (missed depressed): {len(false_negatives)}')
    print(f'Total False Positives (false alarms):     {len(false_positives)}')

    print(f'\n---FALSE NEGATIVES (depressed posts model MISSED) ---')
    for i, row in false_negatives.head(n).iterrows():
        print(f'\nPost {i+1}:')
        print(f'  {row["text_minimal"]}')  #human readable version

    print(f'\n--- FALSE POSITIVES (not depressed posts wrongly flagged) ---')
    for i, row in false_positives.head(n).iterrows():
        print(f'\nPost {i+1}:')
        print(f'  {row["text_minimal"]}')

In [ ]:
# False negatives for all four models


models_to_analyze = {
    'Logistic Regression': (lr_tfidf_bi, X_dev_tfidf_bi),
    'SVM': (svm_model, X_dev_tfidf_bi),
    'Random Forest': (rf_model, X_dev_tfidf_bi),
    'Naive Bayes': (nb_tfidf_bi, X_dev_tfidf_bi)
}

for model_name, (model, X_dev) in models_to_analyze.items():
    preds = model.predict(X_dev)

    df = dev.copy().reset_index(drop=True)
    df['predicted'] = preds

    missed = df[
        (df['labels'] == 'depressed') &
        (df['predicted'] == 'not depressed')
    ]

    print(f'\nMODEL: {model_name}')
    print(f'Total missed: {len(missed)} out of 600 depressed posts')
    print()

    for i, row in missed.iterrows():
        print(f'Post {i+1}:')
        print(f'{row["text_minimal"]}')
        print()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')